# Demo 1: Parallel Computing Practice

**Objective:** Experience the power of parallel computing by analyzing patient data with `multiprocessing.Pool`, observe real speedup, and create your first SLURM script.

## Step 1: Local Parallel Analysis (10 minutes)

### Create Sample Patient Data Files

In [ ]:
import pandas as pd
import numpy as np
import os

# DEBUG MODE: Set to True for faster testing
DEBUG_MODE = False  # Change to False for full analysis

# Configure based on mode
if DEBUG_MODE:
    n_files = 4  # Only 4 files for quick testing
    min_patients = 100  # Smaller cohorts
    max_patients = 500
    print("🐣 DEBUG MODE: Creating baby-sized dataset for quick testing...")
else:
    n_files = 2000  # Full 20 files
    min_patients = 1000  # Full-size cohorts
    max_patients = 5000
    print("🚀 FULL MODE: Creating complete dataset...")

# Create directory for patient data
os.makedirs("patient_cohorts", exist_ok=True)

# Generate sample patient cohort files
for i in range(n_files):
    n_patients = np.random.randint(min_patients, max_patients)

    # Simulate patient data
    patients = pd.DataFrame({
        "patient_id": range(n_patients),
        "age": np.random.normal(65, 15, n_patients),
        "risk_score": np.random.beta(
            2, 5, n_patients
        ),  # Most patients low risk
        "hospital_days": np.random.poisson(3, n_patients),
        "comorbidities": np.random.poisson(2, n_patients),
    })

    patients.to_csv(f"patient_cohorts/cohort_{i:02d}.csv", index=False)

print(f"Created {n_files} patient cohort files!")

🚀 FULL MODE: Creating complete dataset...
Created 2000 patient cohort files!


### Define the Analysis Function

First, let's define the core function that will perform our simulated analysis on a single patient cohort file. This function reads a CSV, simulates some work with `time.sleep()`, and returns a dictionary of results.

In [ ]:
import pandas as pd  # Make sure pandas is imported for this cell
import time  # For time.sleep() and timing

# Ensure DEBUG_MODE is defined. If this cell is run independently,
# you might need to define it here or ensure the cell above (data generation) has been run.
# For safety, let's check and define if not present.
if "DEBUG_MODE" not in globals():
    print("DEBUG_MODE not found, setting to True for this cell.")
    DEBUG_MODE = True


def analyze_patient_cohort(patient_file_path):
    """
    Analyzes a single patient cohort CSV file.

    Args:
        patient_file_path (str): The path to the patient cohort CSV file.

    Returns:
        dict: A dictionary containing summary statistics and processing time.
    """
    df = pd.read_csv(patient_file_path)

    # Simulate complex analysis - shorter sleep in debug mode
    # DEBUG_MODE needs to be accessible here.
    sleep_duration = 0.01 if DEBUG_MODE else 0.1
    time.sleep(sleep_duration)  # Represents actual computation time

    return {
        "file": patient_file_path,
        "n_patients": len(df),
        "avg_age": df["age"].mean(),
        "high_risk_count": (df["risk_score"] > 0.8).sum(),
        "completion_time": time.time(),  # Record completion for potential event ordering analysis (advanced)
    }


# Test the function with one file (optional, good for debugging)
if "current_patient_files" in globals() and current_patient_files:
    print(f"Testing analyze_patient_cohort with: {current_patient_files[0]}")
    single_result = analyze_patient_cohort(current_patient_files[0])
    print(f"Result: {single_result}")
else:
    # Create a dummy file for testing if current_patient_files is not defined
    # This ensures the cell can run even if the data generation cell wasn't executed immediately prior.
    print("Creating a dummy file for testing analyze_patient_cohort...")
    dummy_df = pd.DataFrame({"patient_id": [0], "age": [0], "risk_score": [0]})
    dummy_file_path = "patient_cohorts/cohort_dummy.csv"
    dummy_df.to_csv(dummy_file_path, index=False)
    if (
        "current_patient_files" not in globals()
    ):  # Define if it truly doesn't exist
        current_patient_files = [dummy_file_path]  # Use dummy for the test
    elif not current_patient_files:  # If it exists but is empty
        current_patient_files.append(dummy_file_path)

    print(f"Testing analyze_patient_cohort with: {current_patient_files[0]}")
    single_result = analyze_patient_cohort(current_patient_files[0])
    print(f"Result: {single_result}")


Testing analyze_patient_cohort with: patient_cohorts/cohort_dummy.csv
Result: {'file': 'patient_cohorts/cohort_dummy.csv', 'n_patients': 1, 'avg_age': np.float64(0.0), 'high_risk_count': np.int64(0), 'completion_time': 1749057203.66134}


### Sequential Processing: The Baseline

Before we jump into parallelism, let's run our analysis sequentially (one file after another) to establish a baseline performance. This is how code runs by default if you don't do anything special.

In [ ]:
import time  # For timing
# Ensure current_patient_files and analyze_patient_cohort are available from previous cells.

if "current_patient_files" not in globals() or not current_patient_files:
    print(
        "⚠️ current_patient_files not defined or empty. Please run the data generation and function definition cells first."
    )
    sequential_time = 0
    sequential_results = []
else:
    print(
        f"⏳ Running sequential analysis for {len(current_patient_files)} files..."
    )
    start_time_seq = time.time()
    sequential_results = [
        analyze_patient_cohort(f) for f in current_patient_files
    ]
    sequential_time = time.time() - start_time_seq
    print(f"✅ Sequential analysis complete in: {sequential_time:.3f}s")

# Store this time for later comparison
# We use a dictionary to store times for different worker counts
execution_times = {"sequential": sequential_time}

⏳ Running sequential analysis for 1 files...
✅ Sequential analysis complete in: 0.106s


### Parallel Processing with `concurrent.futures`

Now, let's use the `concurrent.futures` module, which provides a high-level interface for asynchronously executing callables. We'll primarily use `ThreadPoolExecutor` for notebook robustness, but the structure allows for `ProcessPoolExecutor` in script environments.

We will test with different numbers of `max_workers`. `os.cpu_count()` gives the number of CPUs on your system, which is often a good starting point for `max_workers` in CPU-bound tasks when using `ProcessPoolExecutor`. For `ThreadPoolExecutor`, the optimal number can vary (especially for I/O-bound tasks), but testing different values is instructive. We'll cap our "full" test at 16 workers for this demo.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import os
import sys
import time

# Ensure current_patient_files, analyze_patient_cohort, DEBUG_MODE, and execution_times are available.
if (
    "current_patient_files" not in globals()
    or "analyze_patient_cohort" not in globals()
    or "DEBUG_MODE" not in globals()
    or "execution_times" not in globals()
):
    print(
        "⚠️ One or more required variables (current_patient_files, analyze_patient_cohort, DEBUG_MODE, execution_times) not defined. Please run previous cells."
    )
else:
    is_notebook_environment = "get_ipython" in globals() or hasattr(sys, "ps1")

    # Determine "full" parallelism - os.cpu_count() or 16, whichever is less for demo purposes
    # In DEBUG_MODE, we might want to limit this further to keep tests very fast.
    default_max_workers = (
        os.cpu_count() or 1
    )  # Fallback to 1 if cpu_count is None or 0
    sensible_max_workers = min(
        default_max_workers, 16
    )  # Cap at 16 for the demo
    if DEBUG_MODE:
        sensible_max_workers = min(
            sensible_max_workers, 4
        )  # Further cap in debug mode

    worker_configs = [sensible_max_workers, 8, 4]  # Test full, 8, 4
    # Ensure we only test valid worker counts (e.g., don't test 8 if sensible_max_workers is 4)
    # Also, ensure unique values and sort them in descending order for a nice printout.
    worker_configs = sorted(
        list(
            set([
                w for w in worker_configs if w <= sensible_max_workers and w > 0
            ])
        ),
        reverse=True,
    )
    if (
        not worker_configs and sensible_max_workers > 0
    ):  # if list is empty but sensible_max_workers is valid
        worker_configs.append(sensible_max_workers)
    if (
        not worker_configs and sensible_max_workers == 0
    ):  # if sensible_max_workers is 0
        worker_configs.append(
            1
        )  # Default to 1 worker if no other config is valid

    print(f"⚙️ Determined sensible max workers for demo: {sensible_max_workers}")
    print(f"🧪 Will test with worker counts: {worker_configs}")

    for n_workers in worker_configs:
        print(f"\n⚡ Running parallel analysis with {n_workers} workers...")
        start_time_par = time.time()
        parallel_results = []
        executor_used_desc = "None"

        if is_notebook_environment:
            print(
                f"🔧 Using ThreadPoolExecutor for {n_workers} workers (notebook-friendly)."
            )
            try:
                with ThreadPoolExecutor(max_workers=n_workers) as executor:
                    parallel_results = list(
                        executor.map(
                            analyze_patient_cohort, current_patient_files
                        )
                    )
                executor_used_desc = f"ThreadPoolExecutor({n_workers})"
            except Exception as e_thread:
                print(
                    f"⚠️ ThreadPoolExecutor failed ({e_thread}), falling back to sequential for this run..."
                )
                parallel_results = [
                    analyze_patient_cohort(f) for f in current_patient_files
                ]  # Run sequentially as fallback
                executor_used_desc = (
                    f"Sequential Fallback (Thread {n_workers} failed)"
                )
        else:  # Script environment
            print(
                f"🚀 Using ProcessPoolExecutor for {n_workers} workers (script environment)."
            )
            try:
                with ProcessPoolExecutor(max_workers=n_workers) as executor:
                    parallel_results = list(
                        executor.map(
                            analyze_patient_cohort, current_patient_files
                        )
                    )
                executor_used_desc = f"ProcessPoolExecutor({n_workers})"
            except Exception as e_proc:
                print(
                    f"⚠️ ProcessPoolExecutor failed ({e_proc}), falling back to sequential for this run..."
                )
                parallel_results = [
                    analyze_patient_cohort(f) for f in current_patient_files
                ]  # Run sequentially as fallback
                executor_used_desc = (
                    f"Sequential Fallback (Process {n_workers} failed)"
                )

        current_parallel_time = time.time() - start_time_par
        execution_times[
            f"parallel_{n_workers}_workers ({executor_used_desc.split('(')[0]})"
        ] = current_parallel_time
        print(
            f"✅ Parallel analysis with {n_workers} workers ({executor_used_desc}) complete in: {current_parallel_time:.3f}s"
        )

    # Print summary of times
    print("\n\n📊 Execution Time Summary:")
    print(f"{'-' * 30}")
    if "sequential" in execution_times:
        seq_time = execution_times["sequential"]
        print(f"Sequential (1 worker):      {seq_time:.3f}s")
        for config, time_taken in execution_times.items():
            if config != "sequential":
                speedup = (
                    seq_time / time_taken if time_taken > 0 else float("inf")
                )
                print(
                    f"{config:<28}: {time_taken:.3f}s (Speedup: {speedup:.2f}x)"
                )
    else:
        print("Sequential time not recorded. Please run the sequential cell.")
    print(f"{'-' * 30}")


⚙️ Determined sensible max workers for demo: 10
🧪 Will test with worker counts: [10, 8, 4]

⚡ Running parallel analysis with 10 workers...
🔧 Using ThreadPoolExecutor for 10 workers (notebook-friendly).
✅ Parallel analysis with 10 workers (ThreadPoolExecutor(10)) complete in: 0.102s

⚡ Running parallel analysis with 8 workers...
🔧 Using ThreadPoolExecutor for 8 workers (notebook-friendly).
✅ Parallel analysis with 8 workers (ThreadPoolExecutor(8)) complete in: 0.106s

⚡ Running parallel analysis with 4 workers...
🔧 Using ThreadPoolExecutor for 4 workers (notebook-friendly).
✅ Parallel analysis with 4 workers (ThreadPoolExecutor(4)) complete in: 0.104s


📊 Execution Time Summary:
------------------------------
Sequential (1 worker):      0.106s
parallel_10_workers (ThreadPoolExecutor): 0.102s (Speedup: 1.04x)
parallel_8_workers (ThreadPoolExecutor): 0.106s (Speedup: 1.01x)
parallel_4_workers (ThreadPoolExecutor): 0.104s (Speedup: 1.02x)
------------------------------


---

### Alternative: `multiprocessing.Pool` (Primarily for Standalone `.py` Scripts)

The following cell demonstrates using the original `multiprocessing.Pool`. **This method often causes issues in Jupyter Notebooks (especially on Windows/macOS without careful handling) due to how functions are pickled and how `__main__` is handled.** It's included for completeness and is best run by saving the notebook content as a `.py` file and executing it from the terminal.

**Do NOT run this cell directly in the notebook if you encounter pickling errors.**

In [ ]:
# THIS CELL IS INTENDED FOR .py SCRIPT EXECUTION AND EXPLANATION.
# It will NOT run correctly if executed directly in a Jupyter Notebook cell by nbconvert
# due to pickling issues with functions defined in __main__ of a notebook.
# To make this runnable by nbconvert, it would need to be in its own .py file and called.

if False:  # This 'if False' ensures nbconvert --execute skips this block
    import multiprocessing as mp
    import pandas as pd  # Assuming pandas is needed for analyze_patient_cohort
    import time  # Assuming time is needed
    import os  # Assuming os is needed (e.g. for file paths)

    # Ensure DEBUG_MODE and analyze_patient_cohort are defined here if running as a standalone script.
    # For this example, we'll assume they would be copied from above.
    # Example:
    # DEBUG_MODE = True
    # def analyze_patient_cohort(patient_file_path):
    #     df = pd.read_csv(patient_file_path)
    #     sleep_time = 0.01 if DEBUG_MODE else 0.1
    #     time.sleep(sleep_time)
    #     return { 'file': patient_file_path, 'n_patients': len(df) }

    if (
        __name__ == "__main__"
    ):  # CRITICAL for multiprocessing.Pool on Windows/macOS
        # This code would be part of your run_mp_pool.py
        # Define n_files_mp and patient_files_mp appropriately
        n_files_mp = 4 if DEBUG_MODE else 20
        patient_files_mp = [
            f"patient_cohorts/cohort_{i:02d}.csv" for i in range(n_files_mp)
        ]

        print(
            f"📊 (mp.Pool Example) Analyzing {len(patient_files_mp)} cohort files..."
        )

        start_time_par_mp = time.time()
        try:
            # analyze_patient_cohort must be picklable (e.g. top-level in a .py file)
            with mp.Pool(processes=4) as pool:
                parallel_results_mp = pool.map(
                    analyze_patient_cohort, patient_files_mp
                )
            parallel_time_mp = time.time() - start_time_par_mp
            print(f"Parallel (mp.Pool): {parallel_time_mp:.2f}s")
        except Exception as e:
            print(f"Error with mp.Pool: {e}")
            print(
                "This often happens in notebooks. Ensure this code is in a .py file and analyze_patient_cohort is defined at the top level."
            )


## Step 2: SLURM Script Creation (5 minutes)

Create `my_health_analysis.sh`:

In [17]:
%%bash
#!/bin/bash
#SBATCH --job-name=my_first_health_analysis
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=1:00:00
#SBATCH --output=logs/health_analysis_%j.out

echo "Starting health data analysis on $(hostname)"
echo "Number of CPUs: $SLURM_CPUS_PER_TASK"
echo "Memory allocated: $SLURM_MEM_PER_NODE MB"

# Load modules (if on Wynton)
# module load python/3.9

# Run your parallel analysis
python parallel_patient_analysis.py
echo "Analysis complete!"

Starting health data analysis on DEB-5670H9F-LT
Number of CPUs: 
Memory allocated:  MB


python: can't open file '/Users/christopher/Documents/datasci_223/lectures/10/demo/parallel_patient_analysis.py': [Errno 2] No such file or directory


Analysis complete!


## Success Criteria

✅ **Speedup Observed:** Parallel version runs 2-4x faster than sequential
✅ **Understanding Gained:** Can explain when to use processes vs threads
✅ **SLURM Ready:** Created a working SLURM script for cluster submission
✅ **Resource Awareness:** Understand relationship between data size and memory requirements

> 💡 **Debug Tip:** Start with `DEBUG_MODE = True` to quickly test your setup, then switch to `DEBUG_MODE = False` to see real performance differences with larger datasets.

## Common Issues & Solutions

- **"No speedup observed":** Check if analysis function is CPU-bound. Add more computation if needed.
- **"Runs slower with more processes":** Overhead domination. Try smaller datasets or more computation per task.
- **"Memory errors":** Reduce sample data size or increase memory allocation in SLURM script.
- **"Can't get attribute 'analyze_patient_cohort'":** Jupyter notebook limitation! Use the `concurrent.futures` version above or save as `.py` file.
- **"RuntimeError about bootstrapping":** You forgot the `if __name__ == '__main__':` guard! This is required on macOS/Windows.
- **"NameError: name 'DEBUG_MODE' is not defined":** Make sure DEBUG_MODE is defined at the global level, not inside the main block.

## 💡 Pro Tips

- **In Jupyter:** Use `ThreadPoolExecutor` or `ProcessPoolExecutor` from `concurrent.futures`
- **In Scripts:** Use `multiprocessing.Pool` for maximum performance
- **For I/O tasks:** Threading often works just as well as multiprocessing
- **For CPU tasks:** True multiprocessing gives the best speedup